In [0]:
CREATE OR REPLACE PROCEDURE dbw_healthcare_de_dev.etl.usp_start_stage_audit
(
    IN p_file_id STRING,
    IN p_adf_run_id STRING,
    IN p_stage STRING
)
LANGUAGE SQL
SQL SECURITY INVOKER
AS
BEGIN

    INSERT INTO dbw_healthcare_de_dev.etl.file_stage_audit (
        audit_id,
        file_id,
        batch_id,
        adf_run_id,
        processing_stage,
        attempt_number,
        stage_start_ts,
        stage_status,
        created_at
    )
    SELECT
        UUID(),
        fc.file_id,
        fc.batch_id,
        p_adf_run_id,
        p_stage,
        COALESCE(
            (
                SELECT MAX(a.attempt_number)
                FROM dbw_healthcare_de_dev.etl.file_stage_audit a
                WHERE a.file_id = p_file_id
                  AND a.processing_stage = p_stage
            ), 0
        ) + 1,
        CURRENT_TIMESTAMP(),
        'IN_PROGRESS',
        CURRENT_TIMESTAMP()
    FROM dbw_healthcare_de_dev.etl.file_control fc
    WHERE fc.file_id = p_file_id
      AND fc.adf_run_id = p_adf_run_id
      AND fc.pipeline_status = 'IN_PROGRESS'
      AND fc.processing_stage = p_stage
      AND p_stage IN ('BRONZE', 'SILVER', 'GOLD')
      AND NOT EXISTS (
          SELECT 1
          FROM dbw_healthcare_de_dev.etl.file_stage_audit a
          WHERE a.file_id = fc.file_id
            AND a.adf_run_id = p_adf_run_id
            AND a.processing_stage = p_stage
            AND a.stage_status = 'IN_PROGRESS'
      );

END;